# grads-dict-accumulate-parents — faded example 2: Faded: propagate_node iterates recipe.parents and accumulates

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `grads-dict-accumulate-parents`. The last cell reports your progress on the `Backprop: grads dict accumulate parents` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: grads dict accumulate parents` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`grads-dict-accumulate-parents`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "grads-dict-accumulate-parents"
DD_SUBTOPIC = "Backprop: grads dict accumulate parents"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The `propagate_node` function is the per-node step of the reverse pass. It reads `grads[node]` (the upstream gradient already computed for this node), then iterates `node.recipe.parents.items()` to produce one contribution per parent argument. Each contribution is computed by the matching backward function from `BACK_FUNCS`, then added to the parent's running total in `grads` via the same `get-default-0 + add` pattern.

## Faded exercise 2

Complete `propagate_node`. The blank is the two-line body of the loop: look up the backward function and dispatch it to get the contribution, then accumulate into `grads[parent]`.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t

t.manual_seed(0)

class Recipe:
    def __init__(self, func, args, parents):
        self.func = func
        self.args = args
        self.parents = parents

class Node:
    def __init__(self, name, arr):
        self.name = name
        self.array = arr
        self.recipe = None

def propagate_node(node, grads, BACK_FUNCS):
    out_grad = grads[node]
    for argnum, parent in node.recipe.parents.items():
        back_fn = BACK_FUNCS[(node.recipe.func, argnum)]
        contribution = back_fn(out_grad, node.array, *node.recipe.args)
        grads[parent] = grads.get(parent, 0) + contribution

def mul_back_arg0(out_grad, out, a, b):
    return out_grad * b
def mul_back_arg1(out_grad, out, a, b):
    return out_grad * a

BACK_FUNCS = {('mul', 0): mul_back_arg0, ('mul', 1): mul_back_arg1}

w = Node('w', t.tensor(2.0))
x = Node('x', t.tensor(3.0))
z = Node('z', t.tensor(6.0))
z.recipe = Recipe('mul', (w.array, x.array), {0: w, 1: x})

grads = {z: t.tensor(1.0)}
propagate_node(z, grads, BACK_FUNCS)
print(grads[w], grads[x])  # expect 3.0, 2.0


def _test():
    import torch as t

    class Recipe:
        def __init__(self, func, args, parents):
            self.func = func
            self.args = args
            self.parents = parents

    class Node:
        def __init__(self, name, arr):
            self.name = name
            self.array = arr
            self.recipe = None

    def mul_back_arg0(out_grad, out, a, b): return out_grad * b
    def mul_back_arg1(out_grad, out, a, b): return out_grad * a
    BF = {('mul', 0): mul_back_arg0, ('mul', 1): mul_back_arg1}

    # Test 1: distinct parents
    w = Node('w', t.tensor(2.0))
    x = Node('x', t.tensor(3.0))
    z = Node('z', t.tensor(6.0))
    z.recipe = Recipe('mul', (w.array, x.array), {0: w, 1: x})
    grads = {z: t.tensor(1.0)}
    propagate_node(z, grads, BF)
    assert t.allclose(grads[w], t.tensor(3.0)), f"w grad: {grads[w]}"
    assert t.allclose(grads[x], t.tensor(2.0)), f"x grad: {grads[x]}"

    # Test 2: same parent in both argnum slots (y = w * w)
    w2 = Node('w2', t.tensor(4.0))
    y = Node('y', t.tensor(16.0))
    y.recipe = Recipe('mul', (w2.array, w2.array), {0: w2, 1: w2})
    grads2 = {y: t.tensor(1.0)}
    propagate_node(y, grads2, BF)
    # d(w^2)/dw = 2w = 8
    assert t.allclose(grads2[w2], t.tensor(8.0)), f"w2 grad: {grads2[w2]}"


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

t.manual_seed(0)

class Recipe:
    def __init__(self, func, args, parents):
        self.func = func
        self.args = args
        self.parents = parents

class Node:
    def __init__(self, name, arr):
        self.name = name
        self.array = arr
        self.recipe = None

def propagate_node(node, grads, BACK_FUNCS):
    out_grad = grads[node]
    for argnum, parent in node.recipe.parents.items():
        back_fn = BACK_FUNCS[(node.recipe.func, argnum)]
        contribution = back_fn(out_grad, node.array, *node.recipe.args)
        grads[parent] = grads.get(parent, 0) + contribution

def mul_back_arg0(out_grad, out, a, b):
    return out_grad * b
def mul_back_arg1(out_grad, out, a, b):
    return out_grad * a

BACK_FUNCS = {('mul', 0): mul_back_arg0, ('mul', 1): mul_back_arg1}

w = Node('w', t.tensor(2.0))
x = Node('x', t.tensor(3.0))
z = Node('z', t.tensor(6.0))
z.recipe = Recipe('mul', (w.array, x.array), {0: w, 1: x})

grads = {z: t.tensor(1.0)}
propagate_node(z, grads, BACK_FUNCS)
print(grads[w], grads[x])  # expect 3.0, 2.0
```
</details>